In [9]:
import json
import pandas as pd

# Load files
with open("semantic_relevance_results.json", "r", encoding="utf-8") as f:
    relevance = json.load(f)

with open("coverage_results_rich.json", "r", encoding="utf-8") as f:
    completeness = json.load(f)

with open("dataset_sample_10k.json", "r", encoding="utf-8") as f:
    datasets = json.load(f)

# DataFrames

df_rel = pd.DataFrame(relevance)

if isinstance(completeness, dict) and "dataset_results" in completeness:
    df_comp = pd.DataFrame(completeness["dataset_results"])
else:
    df_comp = pd.DataFrame(completeness)

df_data = pd.DataFrame(datasets)

# Metadata

df_data = df_data[
    [
        "dataset_id",
        "title",
        "description",
        "keywords"
    ]
]

# Merge metadata

rel_full = pd.merge(
    df_rel,
    df_data,
    on="dataset_id",
    how="left"
)

comp_full = pd.merge(
    df_comp,
    df_data,
    on="dataset_id",
    how="left"
)

# Export helper

def export_csv(df, filename):
    cols = [c for c in [
        "dataset_id",
        "title",
        "keywords",
        "agg_title_similarity",
        "agg_description_similarity",
        "title_coverage_0.7",
        "description_coverage_0.7"
    ] if c in df.columns]

    df[cols].to_csv(filename, index=False)

# HIGH RELEVANCE

high_rel = (
    rel_full
    .sort_values("agg_title_similarity", ascending=False)
    .head(30)
)

export_csv(
    high_rel,
    "high_relevance_examples.csv"
)

# LOW RELEVANCE

low_rel = (
    rel_full
    .sort_values("agg_title_similarity", ascending=True)
    .head(30)
)

export_csv(
    low_rel,
    "low_relevance_examples.csv"
)

# HIGH COMPLETENESS

high_comp = (
    comp_full
    .sort_values("title_coverage_0.7", ascending=False)
    .head(30)
)

export_csv(
    high_comp,
    "high_completeness_examples.csv"
)

# LOW COMPLETENESS

low_comp = (
    comp_full
    .sort_values("title_coverage_0.7", ascending=True)
    .head(30)
)

export_csv(
    low_comp,
    "low_completeness_examples.csv"
)

# Merge relevance + completeness

merged = pd.merge(
    rel_full,
    comp_full,
    on="dataset_id",
    suffixes=("_rel", "_comp")
)

# HIGH RELEVANCE + LOW COMPLETENESS

high_rel_low_comp = merged[
    (merged["agg_title_similarity"] >= 0.60)
    &
    (merged["title_coverage_0.7"] <= 0.20)
]

high_rel_low_comp = high_rel_low_comp.sort_values(
    "agg_title_similarity",
    ascending=False
)

high_rel_low_comp[
    [
        "dataset_id",
        "title_rel",
        "keywords_rel",
        "agg_title_similarity",
        "title_coverage_0.7"
    ]
].head(30).to_csv(
    "high_relevance_low_completeness.csv",
    index=False
)

# LOW RELEVANCE + HIGH COMPLETENESS


low_rel_high_comp = merged[
    (merged["agg_title_similarity"] <= 0.30)
    &
    (merged["title_coverage_0.7"] >= 0.60)
]

low_rel_high_comp = low_rel_high_comp.sort_values(
    "title_coverage_0.7",
    ascending=False
)

low_rel_high_comp[
    [
        "dataset_id",
        "title_rel",
        "keywords_rel",
        "agg_title_similarity",
        "title_coverage_0.7"
    ]
].head(30).to_csv(
    "low_relevance_high_completeness.csv",
    index=False
)

print("Generated:")
print("- high_relevance_examples.csv")
print("- low_relevance_examples.csv")
print("- high_completeness_examples.csv")
print("- low_completeness_examples.csv")
print("- high_relevance_low_completeness.csv")
print("- low_relevance_high_completeness.csv")

Generated:
- high_relevance_examples.csv
- low_relevance_examples.csv
- high_completeness_examples.csv
- low_completeness_examples.csv
- high_relevance_low_completeness.csv
- low_relevance_high_completeness.csv
